<a href="https://colab.research.google.com/github/cristinalopez1163/03MIAR---Algoritmos-de-Optimizacion/blob/main/SEMINARIO/Seminario_Algoritmos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: Eric Serret Llopis y Cristina López Blanco  <br>
Url:
- GitHub Eric: https://github.com/serretd5/Algoritmos_de_optimizacion_AI_masters_degree
- GitHub Cristina: https://github.com/cristinalopez1163/03MIAR---Algoritmos-de-Optimizacion<br>

Problema:
> **1. Sesiones de doblaje** <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

Descripción del problema: **1. Sesiones de doblaje**

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientementedel número de tomas que se graben. No es posible grabar más  de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible.
- Número de actores: 10
- Número de tomas : 30

(*) La respuesta es obligatoria





                                        

**Carga y limpieza de los datos**

Para resolver este problema se parte de un archivo Excel que contiene la matriz de participación de los actores en las distintas tomas.

Dado que el modelo únicamante requiere la matriz binaria de participación, se limpiarán los datos eliminando:

*   Filas completamente vacías
*   Columnas completamente vacías
* Filas y columnas de totales ya que contienen información agregada

Además, la columna "Toma" se convertirá en índice

In [ ]:
# Carga de las librerías necesarias
import pandas as pd
import numpy as np
from sympy import bell
import math
import random
import copy
import itertools

In [ ]:
from google.colab import drive

file_path = './Datos problema doblaje(30 tomas, 10 actores).xlsx'
df = pd.read_excel(file_path, header = 1)

# Eliminamos las filas y columnas completamente vacías
df = df.dropna(how="all")
df = df.dropna(axis=1, how="all")

# Eliminamos las filas y columnas de totales
df = df.drop(columns=["Total"])
df = df[df["Toma"] != "TOTAL"]

# Columna "Toma" como índice
df = df.set_index("Toma")

display(df)
matriz = df.values

,1,2,3,4,5,6,7,8,9,10
Toma,,,,,,,,,,
1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
5,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
6,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
7,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
8,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


### (*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



### ¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




**Número de posibilidades sin tener en cuenta las restricciones**

Disponemos de la siguiente información:

*  $n=30$ tomas
*  Cada toma puede asignarse a cualquier día
*  No hay límites de tomas por día
*  No hay restricciones de coincidencia de actores

Bajo estas condiciones, cada toma puede agruparse con cualquier otra en un mismo día. El poblema consiste en determinar de cuántas formas distintas pueden agruparse 30 tomas en días (el número de días variará entre 1 y 30).

Para realizar este cálculo nos basamos en el número de Bell, que cuenta en número de particiones posibles en un conjunto de tamaño $n$.

<u>Ejemplo ilustrativo:</u>

Consideramos que el número de tomas $n=3$. El número de Bell correspondiente es $B_3 = 5$:

Las posibles particiones del conjunto $\{\{Toma 1\}, \{Toma 2\}, \{Toma 3\}\}$ son:

- $\{\{Toma\ 1\}, \{Toma\ 2\}, \{Toma\ 3\}\}$
- $\{\{Toma\ 1, Toma\ 2\}, \{Toma\ 3\}\}$
- $\{\{Toma\ 1, Toma\ 3\}, \{Toma\ 2\}\}$
- $\{\{Toma\ 2, Toma\ 3\}, \{Toma\ 1\}\}$
- $\{\{Toma\ 1, Toma\ 2, Toma\ 3\}\}$


In [ ]:
# Número de posibilidades para grabar 30 tomas (sin restricciones)
n = 30
print('El número de opciones posibles para n = ', n, 'tomas es', bell(n))

El número de opciones posibles para n =  30 tomas es 846749014511809332450147


**Número de posibilidades teniendo en cuenta las restricciones**

Las restricciones son:
- Máximo 6 tomas por días (ningún subconjunto de la partición puede tener más de 6 elementos).
- Los actores que comparten escena deben coincidir en la misma sesión (esta restricción no afecta al cálculo de posibilidades).

Dado que tenemos 30 tomas, el número mínimo de días necesarios es $30/6 = 5$ y el número de días máximos es 30 (1 toma por día). La cantidad de posibilidades es la suma de las formas de repartir 30 elementos en $k$ subconjuntos (donde $5\leq k\leq 30$).

El resultado del siguiente código muestra el número de combinaciones posibles para cada organizar las tomas en función del número de días de grabación y suma todos estos valores para obtener el las posibilidades totales.

In [ ]:
def posibilidades_con_restriccion(n_tomas, max_por_dia):
    # diccionario para almacenar los resultados intermedios y no recalcular
    # combinaciones para los mismos valores de n, k
    dictionary = {}

    # Función recursiva: Calcula de cuántas formas se pueden repartir
    # n tomas en exactamente k días
    def solve(n, k):
        if n == 0 and k == 0:
            return 1 # Ya no quedan tomas ni días por repartir

        # Si ya calculamos la combinación antes, la devolvemos
        if (n, k) in dictionary:
            return dictionary[(n, k)]

        total_dia = 0

        # Posibles cantidades de tomas por día (desde 1 a máx por día)
        for i in range(1, min(n, max_por_dia) + 1):
            total_dia += math.comb(n - 1, i - 1) * solve(n - i, k - 1)
            # math.comb(n - 1, i - 1): número de formas de elegir qué
            # tomas van en este día
            # solve(n - i, k - 1):número de formas de repartir las tomas
            # restantes en los días restantes --> Recursión
            # Multiplicamos los dos valores para contar todas las combinaciones
            # posibles y lo sumamos al total

        dictionary[(n, k)] = total_dia # guardamos el resultado
        return total_dia

    # Imprimimos los resultados
    print(f"{'Días (k)':<10} | {'Posibilidades'}")
    print("-" * 50)

    total = 0
    # Iteramos por los días posibles
    num_min = int(n_tomas/max_por_dia)
    for k in range(num_min, n_tomas + 1):
        posibilidades = solve(n_tomas, k)
        total += posibilidades # total global
        print(f"{k:<10} | {posibilidades}")

    print("-" * 50)
    print(f"TOTAL: {total}")

posibilidades_con_restriccion(n_tomas = 30, max_por_dia = 6)

Días (k)   | Posibilidades
--------------------------------------------------
5          | 11423951396577720
6          | 27141023727989347176
7          | 1247092322221582801965
8          | 13637343679657420597875
9          | 60705648218644947402885
10         | 139641497056637478658756
11         | 189690129306340866102900
12         | 165547620103335670775575
13         | 98298768115079797990725
14         | 41370666920172554704740
15         | 12718573016277294641475
16         | 2921326844718182525625
17         | 509888446114294672425
18         | 68481448881446796195
19         | 7140703625732251575
20         | 581365118978443800
21         | 37054373083878150
22         | 1847959038090165
23         | 71823359482875
24         | 2157578049900
25         | 49402080000
26         | 843303006
27         | 10359090
28         | 86275
29         | 435
30         | 1
--------------------------------------------------
TOTAL: 726391948970868949621309


Para simplificar la implementación de los algoritmos de planificación, se ha decidido fijar el número de días de rodaje, que corresponde al cálculo del número mínimo de días necesarios según el máximo de tomas por día permitido. Esto significa que, independientemente de cómo se distribuyan las tomas entre los actores, el algoritmo siempre organizará las sesiones en x días, evitando considerar más de los estrictamente necesarios.

### Modelo para el espacio de soluciones<br>
### (*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Al principio, el plan de grabación se representaba como:

```
plan = [[0,3,5],[1,2,6],[4,7]]
```
donde cada sublista representa un día, con las tomas que se graban ese día. Esta representación facilitaba mucho la comprensión de la solución pero no era demasiado eficiente, especialmente para algoritmos como el recocido simulado donde se generan muchos vecinos y se recalculan costes constantemente. Por ello, al final se decidió optar por un array de Numpy:


```
plan = np.array([3, 1, 3, 1, 2, 3, 1, 2])
```
donde cada índice del array corresponde a una toma, y cada valor indica el día en que se grabará esa toma. Por ejemplo, en este caso el día 1 se grabarían las tomas 2, 4 y 7. Esta representación permite acceder instantáneamente al día de cualquier toma, evitando recorrer listas anidadas y facilita intercambiar de forma eficiente las tomas entre días simplemente cambiando valores del array.


### Según el modelo para el espacio de soluciones<br>
### (*)¿Cual es la función objetivo?

### (*)¿Es un problema de maximización o minimización?

**Función objetivo**

Se trata de un problema de **minimización** ya que buscamos reducir el gasto total. Como todos los actores cobran lo mismo y no influye el número de tomas que graben cada día, la forma de reducir el gasto es minimizando el número de desplazamientos por actor. Es decir, tratando de organizar las sesiones para que los actores coincidan en el mayor número de tomas posibles en un mismo día.

La función a optimizar, según nuestro modelo de espacio de soluciones, sería la siguiente:

$$ \min Z =  \sum_{d \in D} \left| \bigcup_{\{t \,:\, p[t]=d\}} A(t) \right|$$
- $p$ el plan de grabación, es decir, el array donde cada índice corresponde a una toma $t$ y el valor $p[t]$ indica el día en que se grabará esa toma.
- $D$ el conjunto de días posibles de grabación.
- $A(t)$ el conjunto de actores necesario para la toma $t$.


Para cada día $d$, tomamos todas las tomas que se grabarán ese día ($t$ tal que
$p[t]=d$) y hacemos la unión de todos los actores que deben participar en esas tomas ($ \bigcup A(t)$). Después contamos cuántos actores diferentes hay en ese día ($|·|$) y sumamos sobre todos los días.


### Diseña un algoritmo para resolver el problema por fuerza bruta

**Funciones auxiliares**

In [ ]:
# Coste de la solución
def coste_total(plan, matriz):
    num_dias = plan.max() + 1
    coste = 0
    for d in range(num_dias):
        tomas_dia = np.where(plan == d)[0]
        actores_dia = np.any(matriz[tomas_dia], axis=0)
        coste += actores_dia.sum()
    return coste

def imprimir_plan(plan, matriz):
    num_dias = plan.max() + 1
    coste_total_plan = 0

    for d in range(num_dias):
        tomas_dia = np.where(plan == d)[0]
        actores_dia = np.any(matriz[tomas_dia], axis=0)
        actores_presentes = np.where(actores_dia)[0] + 1
        coste_dia = actores_dia.sum()
        coste_total_plan += coste_dia

        print(f"Día {d+1}:")
        print(f"  Tomas grabadas: {tomas_dia + 1}")
        print(f"  Actores presentes: {actores_presentes}")
        print(f"  Coste día: {coste_dia}")
        print("-"*40)

    print(f"Coste total de la planificación: {coste_total_plan}")
    return coste_total_plan


**Algoritmo por fuerza bruta**

In [ ]:
def fuerza_bruta(matriz, tomas_por_dia):

    num_tomas = len(matriz)
    tomas = list(range(num_tomas))
    mejor_plan = None
    mejor_coste = float('inf')

    n_grupos = int(np.ceil(num_tomas / tomas_por_dia))

    # Recorremos todas las permutaciones posibles
    for permutacion in itertools.permutations(tomas):
        # Creamos un array que indique el día de cada toma
        plan = np.zeros(num_tomas, dtype=int)
        for i in range(n_grupos):
            start = i * tomas_por_dia
            end = min(start + tomas_por_dia, num_tomas)
            for t in permutacion[start:end]:
                plan[t] = i  # asigna la toma al día correspondiente

        # Calculamos el coste
        coste = coste_total(plan, matriz)
        if coste < mejor_coste:
            mejor_coste = coste
            mejor_plan = plan.copy()

    imprimir_plan(mejor_plan, matriz)
    return mejor_plan

No es viable ejecutar este algoritmo con una matriz de 30 filas debido a su complejidad exponencial. Por este motivo, se prueba más adelante con un subconjunto de los datos.

### Calcula la complejidad del algoritmo por fuerza bruta

Para tratar de calcular la complejidad del algoritmo se debe calcular la complejidad de cada una de sus partes, en este caso, las funciones que actúan dentro del propio algoritmo. Para ello, primero se calcula la complejidad de cada función:

- Función coste_total: esta función recorre cada día y, para cada uno de ellos, evalúa qué actores trabajan. Por tanto, su complejidad es $O(T⋅A)$ , donde  $T$  es el número de tomas y  $A$  el número de actores.

- Función fuerza_bruta:esta es la función principal que genera todas las permutaciones posibles de las tomas y, para cada permutación, calcula el coste total. Como hay $T!$ permutaciones y cada cálculo de coste requiere $O(T⋅A)$, la complejidad total del algoritmo es $O(T!⋅T⋅A) \approx O(T!)$.

Esto significa que la fuerza bruta crece factorialmente con el número de tomas y se vuelve inviable para valores grandes de $T$.

### (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

El algoritmo por fuerza bruta consiste en generar todas las permutaciones posibles de las tomas y calcular el coste total de cada plan. Su complejidad es factorial, lo hace prácticamente inejecutable para un número de tomas grande, como en este caso con 30 tomas y 10 actores.

Para mejorar esta complejidad, se ha diseñado un algoritmo basado en recocido simulado, un método heurístico que permite aproximar soluciones óptimas sin explorar todas las permutaciones posibles. La idea principal es:
- Se genera una solución inicial aleatoria
- Exploración de soluciones vecinas: El algoritmo genera soluciones cercanas a la actual seleccionando una toma al azar y evaluando en qué otro día podría encajar mejor. La idea es moverla a un día donde ya estén presentes la mayoría de los actores que participan en esa toma, de manera que se minimicen los desplazamientos de los actores y, por tanto, se reduzca el coste total. Este proceso permite mejorar gradualmente la planificación sin reorganizar completamente todas las tomas.
- Si la nueva solución mejora el coste, se acepta automáticamente. Si empeora, se acepta con cierta probabilidad que disminuye a medida que baja la “temperatura”, permitiendo así generar un balance entre intensificación y diversificación.
- Descenso gradual de temperatura

Este algoritmo mejora con respecto a la apoximación por fuerza bruta ya que reemplaza la exploración exhaustiva por un proceso iterativo, que permite recorrer de manera eficiente las posibles soluciones y acercarse a óptimos sin evaluar todas las combinaciones posibles.

**Funciones auxiliares**

In [ ]:
# Genera soluciones aleatorias en función del número de tomas y del máximo de
# tomas permitidas al día
def crear_solucion(num_tomas, tomas_por_dia):
    tomas = list(range(num_tomas))
    random.shuffle(tomas)
    plan = np.zeros(num_tomas, dtype=int)
    for i, toma in enumerate(tomas):
        plan[toma] = i // tomas_por_dia
    return plan



def intercambio_tomas(plan, matriz):
    nuevo_plan = plan.copy()
    num_dias = plan.max() + 1

    # Elegimos una toma aleatoria (seleccionamos día al azar y una toma de ese día)
    toma = random.randint(0, len(plan) - 1)
    dia_origen = plan[toma]
    actores_toma = matriz[toma].astype(bool)

    # Inicializamos variables para almacenar el mejor candidato
    mejor_dia = None
    mejor_interseccion = -1

    # Buscamos el mejor día para mover la toma
    for d in range(num_dias):
        if d == dia_origen: # el día origen no lo recorremos
            continue
        tomas_d = np.where(plan == d)[0]
        actores_d = np.any(matriz[tomas_d], axis=0)
        # Número de actores en común con la toma seleccionada
        interseccion = np.sum(actores_d & actores_toma)
        # Seleccionamos el día que tiene más actores en común con la toma seleccionada
        if interseccion > mejor_interseccion:
            mejor_interseccion = interseccion
            mejor_dia = d

    # Intercambiamos la toma original con la toma seleccionada
    tomas_mejor = np.where(plan == mejor_dia)[0]
    toma_intercambio = random.choice(tomas_mejor)
    nuevo_plan[toma], nuevo_plan[toma_intercambio] = nuevo_plan[toma_intercambio], nuevo_plan[toma]

    return nuevo_plan

# Función de probabilidad para aceptar soluciones peores
def probabilidad(T, delta):
    # Cuando pasamos de la fase de diversificación, cada vez aceptamos
    # menos soluciones sub-óptimas
    return random.random() < np.exp(-delta / T)

# Función de descenso de temperatura
def bajar_temperatura(T, alpha=0.9995):
    # El valor de T disminuye a medida que vamos explorando
    return T * alpha

**Recocido simulado**

In [ ]:
def recocido_simulado(matriz, tomas_por_dia, temperatura=1):

    num_tomas = len(matriz)
    iteraciones = 0

    # solución inicial aleatoria
    solucion_referencia = crear_solucion(num_tomas, tomas_por_dia)
    coste_referencia = coste_total(solucion_referencia, matriz)

    mejor_solucion = copy.deepcopy(solucion_referencia)
    mejor_coste = coste_referencia

    while temperatura > 0.0001:
        iteraciones += 1

        # Generamos nueva solución intercambiando dos tomas
        vecina = intercambio_tomas(solucion_referencia, matriz)
        coste_vecina = coste_total(vecina, matriz)

        # Si la nueva solución es mejor, se guarda
        if coste_vecina < mejor_coste:
            mejor_solucion = copy.deepcopy(vecina)
            mejor_coste = coste_vecina

        # Si la solución es peor, se cambia según la probabilidad
         # (depende de T y delta)
        delta = coste_vecina - coste_referencia
        if delta < 0 or probabilidad(temperatura, delta):
            solucion_referencia = vecina
            coste_referencia = coste_vecina

        # Bajamos la temperatura
        temperatura = bajar_temperatura(temperatura)

    print(f"Número total de iteraciones: {iteraciones}\n")

    imprimir_plan(mejor_solucion, matriz)

    return mejor_solucion


In [ ]:
solucion_recocido = recocido_simulado(matriz, tomas_por_dia=6)

Número total de iteraciones: 18417

Día 1:
  Tomas grabadas: [ 1  5  7 11 20 22]
  Actores presentes: [1 2 3 4 5 8]
  Coste día: 6
----------------------------------------
Día 2:
  Tomas grabadas: [14 17 18 19 23 24]
  Actores presentes: [1 3 6]
  Coste día: 3
----------------------------------------
Día 3:
  Tomas grabadas: [ 3  4  8 15 21 29]
  Actores presentes: [1 2 5 6 7 8]
  Coste día: 6
----------------------------------------
Día 4:
  Tomas grabadas: [ 2  6 10 12 26 30]
  Actores presentes: [1 2 3 4 5 6 9]
  Coste día: 7
----------------------------------------
Día 5:
  Tomas grabadas: [ 9 13 16 25 27 28]
  Actores presentes: [ 1  2  4  5 10]
  Coste día: 5
----------------------------------------
Coste total de la planificación: 27


### (*)Calcula la complejidad del algoritmo

Para analizar la complejidad del algoritmo, es necesario estudiar de forma individual cada una de las funciones que lo componen:

*   coste_total: esta función recorre cada día y, para cada uno de ellos, evalúa qué actores trabajan. Por tanto, su complejidad es $O(T⋅A)$, donde $T$ es el número de tomas y $A$ el número de actores.
*   crear_solucion: recorre la lista de tomas una única vez, por lo que su complejidad es $O(T)$.
* intercambio_tomas: busca el mejor día comparando los actores involucrados en todas las tomas, su complejidad también es $O(T \cdot A)$.
* bajar_temperatura y probabilidad: ambas realizan operaciones matemáticas sencillas por lo que su complejidad es $O(1)$.

La función principal del algoritmo es recocido_simulado. En cada iteración del bucle principal se hace una llamada a las funciones intercambio_tomas y coste_total, ambas con complejidad $O(T·A)$. El número de iteraciones viene determinado por los parámetros de temperatura inicial ($T_{ini}$), temperatura mínima ($T_{min}$) y $\alpha$ (este valor se calcula y muestra en el código). En este caso:


$$k = \frac{\log\left(\frac{T_{min}}{T_{ini}}\right)}{\log(\alpha)} = \frac{\log\left(\frac{0.0001}{1}\right)}{\log(0.9995)} \approx 18.417$$

La complejidad total del algoritmo es $O(k · T · A)$, pero como $k$ es constante e independiente del tamaño del problema obtenemos $O(T · A)$

Por tanto, empleando la estrategia de recocido simulado la complejidad del problema se vuelve lineal con respecto al número de tomas y actores.

### Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Para realizar este apartado se crea una matriz aleatoria de tomas y actores, con una probabilidad predefinida de que el actor participe o no en una toma. Se pueden ajustar varios datos de entrada, como el número de tomas (n_tomas), el número de actores (n_actores) así como la posibilidad de que un actor esté presente en una toma en un determinado día. Podría aleatorizarse todavía más añadiendo un factor aleatorio en el número de tomas y en el de actores, pero por simplicidad ha decidido mantenerse así.

In [ ]:
def generar_datos_rodaje(n_tomas=30, n_actores=15, probabilidad_presencia=0.3):

    # Creamos una matriz llena de números aleatorios entre 0 y 1.
    # np.random.rand genera una estructura del tamaño (filas, columnas).
    matriz_aleatoria = np.random.rand(n_tomas, n_actores)

    # Convertimos los números en 0 o 1 basándonos en la probabilidad
    # Si el número aleatorio es menor que la probabilidad_presencia,
    # será True (1), si no False (0) y se transforman esos Booleanos en 1s y 0s.
    matriz = (matriz_aleatoria < probabilidad_presencia).astype(int)

    # Limpieza para evitar tomas vacías, ya que una toma sin actores no
    # tiene sentido en este problema
    for i in range(n_tomas):
        # Si la suma de la fila es 0, significa que no hay nadie
        if np.sum(matriz[i, :]) == 0:
            # Elegimos un actor al azar y lo obligamos a estar en esa toma
            actor_al_azar = np.random.randint(0, n_actores)
            matriz[i, actor_al_azar] = 1

    # Limpieza para evitar actores sin trabajo
    # Un actor que no sale en ninguna toma no genera desplazamientos
    for j in range(n_actores):
        # Si la suma de la columna es 0, el actor no trabaja nunca
        if np.sum(matriz[:, j]) == 0:
            # Elegimos una toma al azar y metemos al actor
            toma_al_azar = np.random.randint(0, n_tomas)
            matriz[toma_al_azar, j] = 1

    return matriz


nueva_matriz = generar_datos_rodaje(n_tomas=30, n_actores=15, probabilidad_presencia=0.25)
print("Ejemplo de matriz generada (primeras 5 tomas):")
print(nueva_matriz[:5, :])
print(f"\nForma de la matriz: {nueva_matriz.shape}")

Ejemplo de matriz generada (primeras 5 tomas):
[[0 1 1 0 0 0 0 0 1 0 0 0 0 0 0]
 [1 0 0 1 0 1 0 1 0 0 0 1 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 1 1 0 0 0 0 0 0 0 0 1 0]
 [0 1 0 1 0 0 1 0 0 0 1 0 1 1 0]]

Forma de la matriz: (30, 15)


### Aplica el algoritmo al juego de datos generado

Para poder aplicar el algoritmo de fuerza bruta, vamos a crear una matriz con  tomas y 6 actores. Cada día pueden grabarse como máximo 3 tomas.

In [ ]:
nuevos_datos = generar_datos_rodaje(n_tomas=9, n_actores=6, probabilidad_presencia=0.4)
print(nuevos_datos)

[[1 0 1 0 1 0]
 [0 0 1 0 1 1]
 [1 0 0 0 0 0]
 [0 0 1 0 1 1]
 [1 1 1 0 1 1]
 [0 1 1 0 0 1]
 [1 0 0 0 1 0]
 [0 0 1 1 0 0]
 [0 1 0 0 0 0]]


In [ ]:
# fuerza bruta
solucion_fuerza_bruta = fuerza_bruta(nuevos_datos, tomas_por_dia=3)

Día 1:
  Tomas grabadas: [1 3 7]
  Actores presentes: [1 3 5]
  Coste día: 3
----------------------------------------
Día 2:
  Tomas grabadas: [2 4 5]
  Actores presentes: [1 2 3 5 6]
  Coste día: 5
----------------------------------------
Día 3:
  Tomas grabadas: [6 8 9]
  Actores presentes: [2 3 4 6]
  Coste día: 4
----------------------------------------
Coste total de la planificación: 12


In [ ]:
# recocido simulado
solucion_recocido = recocido_simulado(nuevos_datos, tomas_por_dia=3)

Número total de iteraciones: 18417

Día 1:
  Tomas grabadas: [2 4 8]
  Actores presentes: [3 4 5 6]
  Coste día: 4
----------------------------------------
Día 2:
  Tomas grabadas: [5 6 9]
  Actores presentes: [1 2 3 5 6]
  Coste día: 5
----------------------------------------
Día 3:
  Tomas grabadas: [1 3 7]
  Actores presentes: [1 3 5]
  Coste día: 3
----------------------------------------
Coste total de la planificación: 12


### Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo


- https://www.wextensible.com/temas/combinatoria/particiones.html

### Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño.

Una posible solución que todavía reduciría más el número de desplazamientos es tratar de que los actores se quedaran alojados en el estudio de grabación hasta el momento en el que terminan todas sus escenas, y no se les cobra por días de grabación sino por desplazamientos. Si hacen esto, los costes se verían muy reducidos, ya que un actor que tiene que grabar el día 1 y el día 3, podría pernoctar también el día 2 en el estudio. Se ha pedido a Gemini que intente ofrecer una solución con esta configuración y el código obtenido ha sido el siguiente:

In [ ]:
def posible_mejora(matriz, max_tomas_dia):
    n_tomas, n_actores = matriz.shape

    # Calculamos qué actores están en cada toma
    actores_por_toma = [set(np.where(matriz[i, :] == 1)[0]) for i in range(n_tomas)]

    memo = {}
    saltos = {}

    def calcular(idx_inicio, actores_en_set_ayer):
        if idx_inicio == n_tomas:
            # Al terminar, todos los actores que estaban en el set se van a casa
            return len(actores_en_set_ayer)

        # El estado ahora depende de la toma y de quién se quedó a dormir
        estado = (idx_inicio, tuple(sorted(actores_en_set_ayer)))
        if estado in memo:
            return memo[estado]

        mejor_coste = float('inf')

        for tam in range(1, min(max_tomas_dia, n_tomas - idx_inicio) + 1):
            tomas_hoy = range(idx_inicio, idx_inicio + tam)

            # Actores que necesitan estar hoy en el set
            actores_hoy = set()
            for t in tomas_hoy:
                actores_hoy.update(actores_por_toma[t])

            # Cálculo de movimientos
            # 1. ¿Quién llega nuevo hoy? (No estaban ayer)
            llegadas = len(actores_hoy - actores_en_set_ayer)

            # 2. ¿Quién se podría ir hoy?
            # (asumimos que si no trabajan mañana, se van)
            # Esto es lo que minimiza los desplazamientos totales

            coste_total = llegadas + calcular(idx_inicio + tam, actores_hoy)

            if coste_total < mejor_coste:
                mejor_coste = coste_total
                saltos[estado] = tam

        memo[estado] = mejor_coste
        return mejor_coste

    # Iniciamos con el set vacío
    coste_total_movimientos = calcular(0, set())

    # Reconstruimos la solución
    particion = []
    curr = 0
    actores_ayer = set()

    while curr < n_tomas:
        estado = (curr, tuple(sorted(actores_ayer)))
        tam = saltos[estado]

        tomas_hoy = list(range(curr, curr + tam))
        particion.append(tomas_hoy)

        actores_hoy = set()
        for t in tomas_hoy:
            actores_hoy.update(actores_por_toma[t])

        actores_ayer = actores_hoy
        curr += tam

    return particion, coste_total_movimientos, actores_por_toma


# Imprimimos los resultados
particion_optima, coste_optimo, actores_por_toma = posible_mejora(matriz, max_tomas_dia=6)

print(f"Coste óptimo de desplazamientos: {coste_optimo}")
print(f"Número de Sesiones: {len(particion_optima)}\n")

actores_ayer = set()

for i, dia in enumerate(particion_optima, 1):

    actores_hoy = set()
    for t in dia:
        actores_hoy.update(actores_por_toma[t])

    # Llegadas del día
    llegadas = len(actores_hoy - actores_ayer)

    # Salidas del día
    salidas = len(actores_ayer - actores_hoy)

    print(f"Día {i}: Sesiones {dia}")
    print(f"Actores presentes: {len(actores_hoy)}")
    print(f"Llegadas hoy: {llegadas}")
    print(f"Salidas hoy: {salidas}")
    print(f"Número de Sesiones acumuladas: {i}\n")

    actores_ayer = actores_hoy

Coste óptimo de desplazamientos: 15
Número de Sesiones: 8

Día 1: Sesiones [0]
Actores presentes: 5
Llegadas hoy: 5
Salidas hoy: 0
Número de Sesiones acumuladas: 1

Día 2: Sesiones [1, 2, 3, 4]
Actores presentes: 7
Llegadas hoy: 2
Salidas hoy: 0
Número de Sesiones acumuladas: 2

Día 3: Sesiones [5, 6, 7, 8, 9, 10]
Actores presentes: 8
Llegadas hoy: 2
Salidas hoy: 1
Número de Sesiones acumuladas: 3

Día 4: Sesiones [11, 12, 13]
Actores presentes: 6
Llegadas hoy: 0
Salidas hoy: 2
Número de Sesiones acumuladas: 4

Día 5: Sesiones [14, 15, 16, 17, 18, 19]
Actores presentes: 8
Llegadas hoy: 2
Salidas hoy: 0
Número de Sesiones acumuladas: 5

Día 6: Sesiones [20, 21, 22, 23, 24, 25]
Actores presentes: 9
Llegadas hoy: 2
Salidas hoy: 1
Número de Sesiones acumuladas: 6

Día 7: Sesiones [26, 27, 28]
Actores presentes: 4
Llegadas hoy: 0
Salidas hoy: 5
Número de Sesiones acumuladas: 7

Día 8: Sesiones [29]
Actores presentes: 2
Llegadas hoy: 0
Salidas hoy: 2
Número de Sesiones acumuladas: 8



Podemos decir que es un poco extraño que, para 10 actores que en algún momento deberán ir y volver a su casa, haya solo 15 soluciones y habría que investigar más adelante si la solución ofrecida por la IA es correcta o no, pero esto da una buena idea de la gran reducción en gastos que este sistema y esta manera de trabajar supondría para el estudio.



# **Uso de IA**
En un principio, y debido a que no teníamos ciertamente una idea clara de cómo comenzar, comenzamos preguntándole a la IA acerca de ideas iniciales para hacer y probar algoritmos de programación dinámica y de fuerza bruta como tal, con prompts que decían exactamente eso: "En base a estos datos del problema, genérame un algoritmo que cumpla estas condiciones por fuerza bruta y/o por programación dinámica".

Sin embargo, este método fue bastante poco exitoso: Aunque en el algoritmo de fuerza bruta sí que fue utilizado para un primer boceto (que después también fue modificado), el algoritmo de programación dinámica obtenido con IA primero no funcionaba, y luego tras modificarlo nosotros mismos para que funcionase, daba un resultado bastante malo, de alrededor de 43 viajes.

Tras esto, se decidió por hacer un algoritmo de recocido simulado de factura propia, que ha terminado siendo una mejor solución.

La IA se ha utilizado, como ya se ha marcado, también en el apartado de posibles mejoras u optimizaciones. La idea principal para este apartado es de cosecha propia, y a la IA simplemente se le preguntó: "Desarrolla una posible implementación de este problema, en la cual en lugar de contar los días de rodaje se cuentan los desplazamientos al estudio".

Este último algoritmo, por cierto, tampoco funcionaba y fue modificado por los integrantes del equipo para que cuadrase, aunque no se ha revisado a fondo así que puede contener algún error y simplemente debe usarse como ejemplo de futuras evoluciones.